# 04 — LSTM: five- and ten-minute forecasts
Requires [00 — Data preparation](00_Data_Preparation.ipynb). Sends its predictions to [05 — Combined results](05_Combined_Results.ipynb).

Direct forecasts at both horizons. The five-minute run is a new extension; the earlier comparison only ran this neural model at ten minutes.

Each setup uses 30 Optuna trials with the earlier parameter ranges, source weights and trial row caps. Test rows never select settings. These test groups have been examined previously, so this is a rerun, not fresh independent confirmation.

## 1. Setup and load the common prepared data

In [ ]:
%pip -q install "scikit-learn==1.6.1" "optuna>=4,<5" "xgboost>=2,<4" joblib

In [ ]:
from google.colab import drive
from pathlib import Path
import sys

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/KC_CO2_Modular_Notebooks")
sys.path.insert(0, str(PROJECT))
RUN_DIR = PROJECT / "runs" / "real_sessions_v1"
RUN_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import json, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from kc_common import *


In [ ]:
MODEL_NAME = "LSTM"
bundle = joblib.load(RUN_DIR / "prepared.joblib")
DATA, STATE, HIST = bundle["DATA"], bundle["STATE"], bundle["HIST"]
run_id = bundle["run_id"]
OUT = RUN_DIR / MODEL_NAME
OUT.mkdir(exist_ok=True)
RESULTS, MODELS, STUDIES, TIMINGS, PREDICTIONS = [], {}, {}, {}, []
np.random.seed(SEED)
random.seed(SEED)


Choose a GPU runtime for this notebook. The model combines an 11-step sensor sequence with the engineered features. It predicts a CO₂ change, which is added to the current reading.

In [ ]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
print("Device:", DEVICE)


## 2. Evaluation and output helpers
MAE and RMSE are in ppm; R² has no unit. Source = ALL means ordinary row-pooled results, without source balancing.

In [ ]:
def add_metrics(label, horizon, approach, split, ids, pred):
    record_predictions(label, horizon, approach, split, ids, pred)
    d = DATA[horizon].iloc[ids]
    actual = d.actual_future.to_numpy()
    for source in ["ALL", *SOURCE_SHARE]:
        mask = np.ones(len(d), dtype=bool) if source == "ALL" else d.source.to_numpy() == source
        a = actual[mask]
        p = np.asarray(pred)[mask]
        RESULTS.append(
            {
                "Model": label,
                "Horizon_min": horizon,
                "Approach": approach,
                "Split": split,
                "Source": source,
                "Rows": len(a),
                "MAE": mean_absolute_error(a, p),
                "RMSE": mean_squared_error(a, p) ** 0.5,
                "R2": r2_score(a, p),
            }
        )


def record_predictions(label, horizon, approach, split, ids, pred):
    d = DATA[horizon].iloc[ids]
    frame = d[["source", "session", "time", "actual_future", "CO2_now"]].copy()
    frame["row_id"] = ids
    frame["predicted_future"] = pred
    frame["Model"] = label
    frame["Horizon_min"] = horizon
    frame["Approach"] = approach
    frame["Split"] = split
    frame["run_id"] = run_id
    PREDICTIONS.append(frame)


def save_outputs():
    pd.DataFrame(RESULTS).to_csv(OUT / "metrics.csv", index=False)
    pd.concat(PREDICTIONS).to_csv(OUT / "predictions.csv.gz", index=False)
    for key, study in STUDIES.items():
        name = "_".join(map(str, key))
        study.trials_dataframe().to_csv(OUT / f"{name}_trials.csv", index=False)
        params = dict(study.best_params)
        params.update(study.best_trial.user_attrs)
        (OUT / f"{name}_settings.json").write_text(json.dumps(params, indent=2))
    pd.DataFrame(
        [
            {
                "Model": k[0],
                "Horizon_min": k[1],
                "Approach": k[2],
                "Study_and_refit_minutes": t / 60,
            }
            for k, t in TIMINGS.items()
        ]
    ).to_csv(OUT / "timings.csv", index=False)
    (OUT / "run_id.txt").write_text(run_id)
    import importlib.metadata

    versions = {
        name: importlib.metadata.version(name)
        for name in ["numpy", "pandas", "scikit-learn", "optuna", "xgboost", "joblib"]
    }
    (OUT / "versions.json").write_text(json.dumps(versions, indent=2))


## 3. Model and tuning helpers
Tune hidden size, layers, dropout, learning rate, weight decay and batch size. Trials train for up to 16 epochs, stopping after three epochs without improvement. Refit on all training rows for the chosen trial’s best epoch count. A single-layer recurrent model does not use recurrent dropout.

In [ ]:
class Recurrent(nn.Module):

    def __init__(self, kind, nstatic, hidden, layers, dropout):
        super().__init__()
        cls = nn.GRU if kind == "GRU" else nn.LSTM
        self.rnn = cls(
            input_size=3,
            hidden_size=hidden,
            num_layers=layers,
            batch_first=True,
            dropout=dropout if layers > 1 else 0,
        )
        self.head = nn.Sequential(nn.Linear(hidden + nstatic, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, seq, static):
        out, _ = self.rnn(seq)
        return self.head(torch.cat((out[:, -1, :], static), dim=1)).squeeze(1)


def nn_arrays(horizon, ids, target_scale):
    st = STATE[horizon]
    d = DATA[horizon]
    return (
        torch.from_numpy(st["Snn"][ids]),
        torch.from_numpy(st["Xnn"][ids]),
        torch.from_numpy((d.target_direct.to_numpy()[ids] / target_scale).astype(np.float32)),
    )


def nn_predict(model, horizon, ids, target_scale):
    model.eval()
    seq, static, _ = nn_arrays(horizon, ids, target_scale)
    loader = DataLoader(TensorDataset(seq, static), batch_size=2048, shuffle=False)
    parts = []
    with torch.no_grad():
        for a, b in loader:
            parts.append(model(a.to(DEVICE), b.to(DEVICE)).cpu().numpy())
    return DATA[horizon].CO2_now.to_numpy()[ids] + np.concatenate(parts) * target_scale


def fit_recurrent(kind, horizon, params, train_ids, val_ids, early_stop):
    torch.manual_seed(SEED)
    d = DATA[horizon]
    st = STATE[horizon]
    target_scale = max(1.0, float(d.target_direct.iloc[train_ids].std()))
    model = Recurrent(
        kind, st["Xnn"].shape[1], params["hidden"], params["layers"], params["dropout"]
    ).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=params["learning_rate"], weight_decay=params["weight_decay"]
    )
    seq, static, target = nn_arrays(horizon, train_ids, target_scale)
    weights = balanced_weights(d.source.iloc[train_ids])
    weights *= len(weights) / weights.sum()
    loader = DataLoader(
        TensorDataset(seq, static, target, torch.from_numpy(weights.astype(np.float32))),
        batch_size=params["batch_size"],
        shuffle=True,
    )
    best = float("inf")
    best_state = None
    best_epoch = 0
    stale = 0
    for epoch in range(params["epochs"]):
        model.train()
        for a, b, y, w in loader:
            a, b, y, w = (x.to(DEVICE) for x in (a, b, y, w))
            optimizer.zero_grad()
            pred = model(a, b)
            loss = (w * torch.abs(pred - y)).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        if early_stop:
            pred = nn_predict(model, horizon, val_ids, target_scale)
            v = d.iloc[val_ids]
            score = score_sources(v.actual_future.to_numpy(), pred, v.source.to_numpy())
            if score < best - 0.0001:
                best = score
                best_epoch = epoch + 1
                stale = 0
                best_state = {
                    k: x.detach().cpu().clone() for k, x in model.state_dict().items()
                }
            else:
                stale += 1
            if stale >= 3:
                break
    if early_stop:
        model.load_state_dict(best_state)
    return (
        model,
        target_scale,
        best if early_stop else None,
        best_epoch if early_stop else params["epochs"],
    )


def run_recurrent(kind, horizon):
    key = (kind, horizon, "direct")
    d = DATA[horizon]
    st = STATE[horizon]
    start = time.time()
    fitids = st["trial_train"]
    valids = st["trial_val"]

    def objective(trial):
        params = {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "layers": trial.suggest_int("layers", 1, 2),
            "dropout": trial.suggest_float("dropout", 0.0, 0.3),
            "learning_rate": trial.suggest_float("learning_rate", 0.0003, 0.003, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-06, 0.001, log=True),
            "batch_size": trial.suggest_categorical("batch_size", [256, 512, 1024]),
            "epochs": 16,
        }
        model, scale, score, best_epoch = fit_recurrent(
            kind, horizon, params, fitids, valids, True
        )
        trial.set_user_attr("best_epoch", best_epoch)
        del model
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        return score

    study = optuna.create_study(
        direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    STUDIES[key] = study
    params = dict(study.best_params)
    params["epochs"] = study.best_trial.user_attrs["best_epoch"]
    final, scale, _, _ = fit_recurrent(kind, horizon, params, st["train"], st["val"], False)
    MODELS[key] = (final, scale)
    TIMINGS[key] = time.time() - start
    for split in ("val", "test"):
        ids = st[split]
        pred = nn_predict(final, horizon, ids, scale)
        add_metrics(
            kind, horizon, "direct", {"val": "Validation", "test": "Test"}[split], ids, pred
        )
    print(
        key,
        "best trial score",
        round(study.best_value, 3),
        "parameters",
        params,
        "elapsed minutes",
        round(TIMINGS[key] / 60, 1),
    )


## 4. Run 10-minute forecasts
Print Local, UPC, Slovenia and ALL scores for validation and test. Results and trial histories are saved after this horizon finishes.

In [ ]:
run_recurrent(MODEL_NAME, 10)
save_outputs()
display(
    pd.DataFrame(RESULTS)
    .query("Horizon_min == 10")
    .sort_values(["Split", "Approach", "Source"])
    .round(3)
)
model, target_scale = MODELS[(MODEL_NAME, 10, "direct")]
torch.save(
    {
        "state_dict": model.state_dict(),
        "target_scale": target_scale,
        "params": STUDIES[(MODEL_NAME, 10, "direct")].best_params,
        "run_id": run_id,
    },
    OUT / "10min_direct.pt",
)


## 5. Run 5-minute forecasts
Print Local, UPC, Slovenia and ALL scores for validation and test. Results and trial histories are saved after this horizon finishes.

In [ ]:
run_recurrent(MODEL_NAME, 5)
save_outputs()
display(
    pd.DataFrame(RESULTS)
    .query("Horizon_min == 5")
    .sort_values(["Split", "Approach", "Source"])
    .round(3)
)
model, target_scale = MODELS[(MODEL_NAME, 5, "direct")]
torch.save(
    {
        "state_dict": model.state_dict(),
        "target_scale": target_scale,
        "params": STUDIES[(MODEL_NAME, 5, "direct")].best_params,
        "run_id": run_id,
    },
    OUT / "5min_direct.pt",
)


## Next step
Run the other model notebooks, then [05 — Combined results](05_Combined_Results.ipynb). Comparing model tables does not retrain or reweight predictions.